Imports, change the directories

In [1]:
# Install Unsloth and dependencies
!pip install -q --upgrade pip

!pip install -q \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install -q \
    transformers \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### 4. Update the Adapter Save Path

Before training, update `OUTPUT_DIR` to the directory where you would like the trained SFT adapter to be saved.

```python
OUTPUT_DIR = "/path/to/sft_adapter"
```

For example:

```python
OUTPUT_DIR = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter"
```

The pretrained SFT adapter generated after training will be saved to this directory. If you are using the provided project structure, the default SFT adapter location can be found under:

```text
/content/drive/MyDrive/colm_grpo_llmaj_dataset/train/llama3b/sft_adapter
```

Subsequent GRPO training notebooks should use this saved SFT adapter as the initialization point by setting `SFT_ADAPTER_PATH` accordingly.


In [17]:


import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig



BASE_MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

TRAIN_CSV = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/train_subset.csv"

OUTPUT_DIR = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter"

MAX_SEQ_LENGTH = 2048

NUM_EPOCHS = 2
LEARNING_RATE = 5e-6
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

In [ ]:

def make_prompt(row):
    return f"""You are an expert cybersecurity answer evaluator.

You will be given a cybersecurity question and three candidate answers.

Your task is to rank the three answers from best to worst based on:
1. Technical correctness
2. Completeness
3. Relevance to the question
4. Clarity and precision
5. Lack of hallucination or misleading information
6. Keywords match

Assign each rubric a score and then sum all 6 rubric scores to find total score for a response and then rank the responses.

Important:
- R1, R2, and R3 are all candidate answers.
- Do not assume the reference answer is always best.
- Judge only based on answer quality.
- Output ONLY the ranking.
- The output format must be exactly like one of these:
R1>R2>R3
R1>R3>R2
R2>R1>R3
R2>R3>R1
R3>R1>R2
R3>R2>R1

Question:
{row["question"]}

R1:
{row["answer"]}

R2:
{row["candidate_llama_3_2_1b_instruct"]}

R3:
{row["candidate_qwen3_32b"]}

Ranking:"""


train_df = pd.read_csv(TRAIN_CSV)

required_cols = [
    "question",
    "answer",
    "candidate_llama_3_2_1b_instruct",
    "candidate_qwen3_32b",
    "gpt_51_judge_ranking",
]

missing = [c for c in required_cols if c not in train_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

train_df = train_df.dropna(subset=required_cols).reset_index(drop=True)

train_df["prompt"] = train_df.apply(make_prompt, axis=1)
train_df["completion"] = train_df["gpt_51_judge_ranking"].astype(str).str.strip()

train_df["text"] = train_df["prompt"] + " " + train_df["completion"]

train_dataset = Dataset.from_pandas(train_df[["text"]])


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,

    fp16=False,
    bf16=True,

    logging_steps=10,
    save_steps=50,
    save_total_limit=2,

    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()


model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved SFT adapter to:", OUTPUT_DIR)
